In [5]:
#[CELL 1] STUDENT CONFIGURATION

from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Bautista"
STUDENT_ID = "TUPM-26-3506"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

TARGET_CLASS = (SEED_DIGIT % 4) + 2
RECORDS_TO_PROCESS = 1000 + ID_SUM

print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records To Process:", RECORDS_TO_PROCESS)
print("Target Class:", f"{TARGET_CLASS}xx")

Seed Digit: 6
ID Sum: 22
Records To Process: 1022
Target Class: 4xx


In [6]:
# [CELL 2] CREATE WORKSPACE

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Workspace:", WORK_DIR.resolve())

Workspace: C:\Users\jacob\Documents\Comprog\Lab6_File_Handling_Bautista\Lab6_File_Handling_Additional_Exercises\web-server-access-log-analysis


In [7]:
#[CELL 3] DATASET PATH

dataset_path = DATA_DIR / "weblog.csv"

filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"

summary_json = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"

backup_json = BACKUP_DIR / summary_json.name

In [8]:
# [CELL 4] DATASET PROCESSING

status_counter = Counter()
url_counter = Counter()

selected_records = []

records_processed = 0
records_in_target = 0

with open(dataset_path, "r", encoding="utf-8") as file:

    reader = csv.DictReader(file)

    for row in reader:

        if records_processed >= RECORDS_TO_PROCESS:
            break

        records_processed += 1

        try:

            ip = row["IP"]
            timestamp = row["Time"]
            url = row["URL"]

            status = row["Status"].strip()

            if not status.isdigit():
                continue

            status_code = int(status)

            status_counter[status_code] += 1

            if status_code // 100 == TARGET_CLASS:

                records_in_target += 1

                url_counter[url] += 1

                selected_records.append({
                    "IP": ip,
                    "Time": timestamp,
                    "URL": url,
                    "Status": status_code
                })

        except KeyError:
            continue

print("Records processed:", records_processed)
print("Records in target class:", records_in_target)

Records processed: 1022
Records in target class: 0


In [9]:
#[CELL 5] TOP 5 URLS

top_five_urls = url_counter.most_common(5)

print("Top 5 URLs")

for url, count in top_five_urls:
    print(url, count)

Top 5 URLs


In [10]:
# [CELL 6] EXPORTING

with open(filtered_csv,
        "w",
        newline="",
        encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=["IP", "Time", "URL", "Status"]
    )

    writer.writeheader()

    for record in selected_records:
        writer.writerow(record)

print("Created:", filtered_csv)

Created: output\Bautista_TUPM-26-3506_weblog_filtered.csv


In [11]:
# [CELL 7] SUMMARY JSON

summary_data = {

    "records_processed": records_processed,

    "target_status_class": f"{TARGET_CLASS}xx",

    "records_in_target_class": records_in_target,

    "count_per_status_code":
        dict(sorted(status_counter.items())),

    "top_five_urls":
        [{"url": u, "count": c}
        for u, c in top_five_urls],

    "filtered_csv_file":
        filtered_csv.name
}

with open(summary_json,
        "w",
        encoding="utf-8") as file:

    json.dump(summary_data,
            file,
            indent=4)

print("Created:", summary_json)

Created: output\Bautista_TUPM-26-3506_weblog_summary.json


In [12]:
# [CELL 8] BACKUP

shutil.copy2(summary_json, backup_json)

print("Original:", summary_json)
print("Backup:", backup_json)
print("Backup Exists:", backup_json.exists())

Original: output\Bautista_TUPM-26-3506_weblog_summary.json
Backup: backup\Bautista_TUPM-26-3506_weblog_summary.json
Backup Exists: True


In [13]:
# [CELL 9] ASSESSMENT OUTPUT

print("\nWEB SERVER ACCESS LOG ANALYSIS")
print("-" * 40)

print("Records Processed:", records_processed)
print("Target Status Class:", f"{TARGET_CLASS}xx")
print("Records In Target Class:", records_in_target)

print("\nCount Per Status Code")

for code, count in sorted(status_counter.items()):
    print(code, ":", count)

print("\nFive Most Requested URLs")

for url, count in top_five_urls:
    print(url, ":", count)

print("\nFiltered CSV:", filtered_csv.name)
print("JSON Summary:", summary_json.name)
print("Backup Verification:", backup_json.exists())


WEB SERVER ACCESS LOG ANALYSIS
----------------------------------------
Records Processed: 1022
Target Status Class: 4xx
Records In Target Class: 0

Count Per Status Code

Five Most Requested URLs

Filtered CSV: Bautista_TUPM-26-3506_weblog_filtered.csv
JSON Summary: Bautista_TUPM-26-3506_weblog_summary.json
Backup Verification: True
